# Demo 1: Predict grouping grain and choose the right count

**Learning question:** How do we predict a grouped result and choose `size`, `count`, or `nunique` from the question being answered?

The input has grain **one recorded encounter per row**. The final table has output grain **one observed facility per row**. This required demo is Colab-first and runs equivalently in local Jupyter or VS Code. Colab storage is ephemeral, and changes opened from GitHub are not automatically saved back to the repository.

Use only the supplied synthetic, non-identifying fixture. Do not add credentials, private records, manual uploads, or Drive mounts. Restart the kernel and run every cell in order; stored output is not execution evidence. Assignment Colab support remains conditional on the repository-save and Classroom50 pilot.


In [ ]:
import platform
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

PYTHON_CANDIDATE = "3.12.13"
NUMPY_CANDIDATE = "2.0.2"
PANDAS_CANDIDATE = "3.0.3"
COURSE_PACKAGES = {"numpy": NUMPY_CANDIDATE, "pandas": PANDAS_CANDIDATE}


def installed_version(package_name):
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None


mismatched = [
    f"{package_name}=={candidate}"
    for package_name, candidate in COURSE_PACKAGES.items()
    if installed_version(package_name) != candidate
]
if mismatched:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *mismatched]
    )

import numpy as np
import pandas as pd

assert platform.python_version() == PYTHON_CANDIDATE
assert np.__version__ == NUMPY_CANDIDATE
assert pd.__version__ == PANDAS_CANDIDATE
print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)


## Define the grouping contract before computing

**Input row grain** says what one source row represents. A **grouping key** is the column whose values determine which rows belong together. A **group** contains rows sharing one observed key value, and the **grouping unit** is the real-world unit that value represents. Here, `facility` is the key and one observed facility is the grouping unit.

**Output row grain** says what one result row represents. An **aggregation** reduces each group to summary values. A pandas **GroupBy object** records how rows are split; it is not itself a summary table.

Prediction before code:

- input row grain: one recorded encounter;
- grouping key: `facility`;
- grouping unit: one observed facility;
- predicted identities: North, South, West;
- predicted number of groups: three;
- category policy: only category values observed in rows; and
- aggregated output grain: one observed facility.


In [ ]:
from hashlib import sha256
from pathlib import Path

EXPECTED_FIXTURE_SHA256 = "24a31904c1371553ff3af627dc21146ed743c8c0c47452ade3628c2fc199c5dc"
FIXTURE_BYTES = (
    b"encounter_id,facility,provider_id,service,charge,wait_minutes,rating\n"
    b"E001,North,P01,Consult,120,20,4\n"
    b"E002,North,P01,Follow-up,80,12,\n"
    b"E003,North,P02,Consult,150,30,5\n"
    b"E004,North,P02,Procedure,210,50,5\n"
    b"E005,South,P03,Consult,110,18,4\n"
    b"E006,South,P03,Consult,90,16,\n"
    b"E007,South,P04,Procedure,220,45,\n"
    b"E008,South,P04,Procedure,125,25,4\n"
    b"E009,West,P05,Consult,130,25,3\n"
    b"E010,West,P05,Procedure,200,40,4\n"
    b"E011,West,P06,Consult,140,35,3\n"
    b"E012,West,P06,Follow-up,75,15,4\n"
)
FACILITY_LEVELS = ["North", "South", "West", "Remote"]
SERVICE_LEVELS = ["Consult", "Follow-up", "Procedure"]


def find_demo_directory(start):
    current = start.resolve()
    while True:
        for candidate in (current, current / "08" / "demo"):
            if (
                (candidate / "DEMO_GUIDE.md").is_file()
                and (candidate / ".python-version").is_file()
            ):
                return candidate
        if current.parent == current:
            return None
        current = current.parent


DEMO_DIRECTORY = find_demo_directory(Path.cwd())
if DEMO_DIRECTORY is None:
    DEMO_DIRECTORY = Path.cwd().resolve()

DATA_DIRECTORY = DEMO_DIRECTORY / "data"
DATA_DIRECTORY.mkdir(parents=True, exist_ok=True)
FIXTURE_PATH = DATA_DIRECTORY / "encounters.csv"
if not FIXTURE_PATH.exists():
    FIXTURE_PATH.write_bytes(FIXTURE_BYTES)

actual_fixture_sha256 = sha256(FIXTURE_PATH.read_bytes()).hexdigest()
assert actual_fixture_sha256 == EXPECTED_FIXTURE_SHA256, (
    "encounters.csv does not match the supplied fixture checksum. "
    "Restore the committed file; corrupt data are never replaced silently."
)

OUTPUT_DIRECTORY = DEMO_DIRECTORY / "output"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIRECTORY / "count_comparison.csv"
if OUTPUT_PATH.exists():
    OUTPUT_PATH.unlink()


def write_repeatable_csv(frame, path, *, na_rep=""):
    frame.to_csv(
        path,
        index=False,
        encoding="utf-8",
        lineterminator="\n",
        na_rep=na_rep,
    )
    first_bytes = path.read_bytes()
    frame.to_csv(
        path,
        index=False,
        encoding="utf-8",
        lineterminator="\n",
        na_rep=na_rep,
    )
    assert path.read_bytes() == first_bytes
    return first_bytes


encounters = pd.read_csv(
    FIXTURE_PATH,
    dtype={
        "encounter_id": "string",
        "provider_id": "string",
        "charge": "int64",
        "wait_minutes": "int64",
        "rating": "Int64",
    },
)
encounters["facility"] = pd.Categorical(
    encounters["facility"],
    categories=FACILITY_LEVELS,
    ordered=True,
)
encounters["service"] = pd.Categorical(
    encounters["service"],
    categories=SERVICE_LEVELS,
    ordered=True,
)

assert encounters.shape == (12, 7)
assert encounters["encounter_id"].is_unique
assert encounters["encounter_id"].dtype == pd.StringDtype()
assert encounters["provider_id"].dtype == pd.StringDtype()
assert encounters["facility"].dtype == pd.CategoricalDtype(
    FACILITY_LEVELS, ordered=True
)
assert encounters["service"].dtype == pd.CategoricalDtype(
    SERVICE_LEVELS, ordered=True
)
assert encounters["charge"].dtype == np.dtype("int64")
assert encounters["wait_minutes"].dtype == np.dtype("int64")
assert encounters["rating"].dtype == pd.Int64Dtype()
assert encounters["rating"].isna().sum() == 3
assert encounters[["facility", "service"]].notna().all().all()

print("Demo directory:", DEMO_DIRECTORY)
print("Fixture SHA-256:", actual_fixture_sha256)
print(encounters)


In [ ]:
facility_groups = encounters.groupby(
    "facility",
    observed=True,
    sort=True,
    dropna=True,
)

facility_sizes = facility_groups.size().rename("encounter_count")
group_identities = [str(value) for value in facility_sizes.index]

assert facility_groups.ngroups == 3
assert group_identities == ["North", "South", "West"]
assert facility_sizes.tolist() == [4, 4, 4]
assert int(facility_sizes.sum()) == len(encounters)
assert "Remote" not in group_identities

print("Observed groups:", group_identities)
print(facility_sizes)


## Match each count to its question

- GroupBy `size()` counts input rows, even when another column is missing: **How many encounters were recorded?**
- Selected-column `count()` counts nonmissing values in that column: **How many encounters have a recorded rating?**
- Selected-column `nunique()` counts distinct nonmissing values: **How many distinct providers appear?**

Lecture 06 already introduced alignment and concatenation. Here `concat` only places three same-index Series beside one another; it is not a new joining lesson.


In [ ]:
count_comparison = pd.concat(
    [
        facility_groups.size().rename("encounter_count"),
        facility_groups["rating"].count().rename("rating_count"),
        facility_groups["provider_id"]
        .nunique(dropna=True)
        .rename("unique_provider_count"),
    ],
    axis="columns",
).reset_index()

assert count_comparison.columns.tolist() == [
    "facility",
    "encounter_count",
    "rating_count",
    "unique_provider_count",
]
assert count_comparison["facility"].astype("string").tolist() == [
    "North",
    "South",
    "West",
]
assert count_comparison["encounter_count"].tolist() == [4, 4, 4]
assert count_comparison["rating_count"].tolist() == [3, 2, 4]
assert count_comparison["unique_provider_count"].tolist() == [2, 2, 2]
assert int(count_comparison["encounter_count"].sum()) == len(encounters)

output_bytes = write_repeatable_csv(count_comparison, OUTPUT_PATH)
EXPECTED_OUTPUT_BYTES = (
    b"facility,encounter_count,rating_count,unique_provider_count\n"
    b"North,4,3,2\n"
    b"South,4,2,2\n"
    b"West,4,4,2\n"
)
assert output_bytes == EXPECTED_OUTPUT_BYTES

count_readback = pd.read_csv(
    OUTPUT_PATH,
    dtype={
        "facility": "string",
        "encounter_count": "int64",
        "rating_count": "int64",
        "unique_provider_count": "int64",
    },
)
expected_count_readback = pd.DataFrame(
    {
        "facility": pd.Series(["North", "South", "West"], dtype="string"),
        "encounter_count": [4, 4, 4],
        "rating_count": [3, 2, 4],
        "unique_provider_count": [2, 2, 2],
    }
)
pd.testing.assert_frame_equal(count_readback, expected_count_readback)

demo1_verified = True
print(count_comparison)
print("Wrote:", OUTPUT_PATH)


## Interpret before continuing

North has four encounter rows, three recorded ratings, and two distinct providers. `size`, `count`, and `nunique` are all correct because they answer different questions; they are not interchangeable. Remote is a declared category but not an observed group because no input row has that facility.


In [ ]:
assert demo1_verified is True
assert sha256(FIXTURE_PATH.read_bytes()).hexdigest() == EXPECTED_FIXTURE_SHA256
assert OUTPUT_PATH.is_file()
assert OUTPUT_PATH.read_bytes() == EXPECTED_OUTPUT_BYTES
assert facility_groups.ngroups == 3
assert int(count_comparison["encounter_count"].sum()) == 12
print("Lecture 08 Demo 1 fresh-execution verification passed.")
